# Presentación de Colab

Obtén más información sobre Google Colab y crea una cuenta de Google (si aún no tienes una) [aquí](https://colab.research.google.com/)

Deberías poder usar el nivel gratuito o el gasto mínimo para completar todos los proyectos de la clase. Personalmente, me registré en Colab Pro+ y me encanta, pero no es obligatorio.

Para ver algunas de las cosas que Colab puede hacer, sigue el siguiente enlace y crea una copia en tu Google Drive.

https://colab.research.google.com/drive/1H9s8jXyfb1cAhXYfUyylNr_qtFhvjxeY?usp=share_link

# Semana 3, día 1

Para este ejemplo de uso de Colab, utilizo una potente GPU para generar una imagen genial.

Para ejecutar este ejemplo, se necesita una A100. Puede costar un par de dólares, así que guardo los resultados aquí para que puedas ver el resultado sin necesidad de usar una A100. Pero si no te importa gastar, ¡es muy divertido! También puedes sustituirla por un modelo más económico si lo prefieres.

In [ ]:
!pip install diffusers transformers accelerate

# Nota importante

No mencioné esto en la conferencia, pero es posible que debas iniciar sesión en el centro de HuggingFace si no lo has hecho antes.

1. Si aún no lo has hecho, crea una cuenta de HuggingFace **gratuita** en https://huggingface.co y navega a Configuración desde el menú de usuario en la parte superior derecha. Luego, crea un nuevo token de API y obtén permisos de escritura.

2. De regreso aquí en colab, presiona el ícono de "llave" en el panel lateral a la izquierda y agrega un nuevo secreto:
En el campo de nombre, coloca `HF_TOKEN`
En el campo de valor, coloca tu token actual: `hf_...`
Asegúrate de que el interruptor de acceso al cuaderno esté encendido.

3. Ejecuta la celda a continuación para iniciar sesión. Deberás hacer esto en cada uno de tus colabs. Es una forma realmente útil de administrar tus secretos sin necesidad de escribirlos en colab. También hay un atajo para sobrescribir la línea que aparece a continuación con:
`hf_token = "hf_...."`
Pero esta no es una buena práctica, ya que deberás tener cuidado de no compartir el colab. ¡Y una de las mejores cosas de los colabs es que puedes compartirlos!

In [ ]:
from huggingface_hub import login
from google.colab import userdata


hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
import torch
from diffusers import FluxPipeline

# OJO: FLUX.1-schnell es un repo "gated" de HuggingFace. Antes de ejecutar esta celda tienes que
# entrar en https://huggingface.co/black-forest-labs/FLUX.1-schnell con tu cuenta y aceptar la
# licencia; si no, la descarga falla con "GatedRepoError: 403 ... you are not in the authorized list".

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16).to("cuda")
generator = torch.Generator(device="cuda").manual_seed(0)
prompt = "Una clase futurista llena de estudiantes que aprenden programación de IA al estilo surrealista de Salvador Dalí"

# Generate the image using the GPU
image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=4,
    max_sequence_length=256,
    generator=generator
).images[0]

image.save("surreal.png")

In [ ]:
from IPython.display import display
from PIL import Image

# Display the image inline
display(image)

In [ ]:
!pip install datasets

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import soundfile as sf
import torch

# El pipeline de una línea `pipeline("text-to-speech", ...)` está roto en la versión actual de
# transformers: su propio código llama a `.to(dtype=...)` sobre el resultado del procesador, y ese
# objeto ya no acepta `dtype` (TypeError: BatchEncoding.to() got an unexpected keyword argument
# 'dtype'). Usamos SpeechT5 directamente, que es lo que el pipeline hacía por dentro.

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to("cuda")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to("cuda")

# `datasets` ya no ejecuta scripts de carga; este dataset tiene una revisión en Parquet que sí funciona
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation",
                                  revision="refs/convert/parquet")
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0).to("cuda")

inputs = processor(text="¡Hola a un ingeniero en inteligencia artificial en camino hacia la maestría!",
                   return_tensors="pt").to("cuda")
speech = model.generate_speech(inputs["input_ids"], speaker_embedding, vocoder=vocoder)

sf.write("speech.wav", speech.cpu().numpy(), samplerate=16000)


In [ ]:
from IPython.display import Audio

# Play the generated audio
Audio("speech.wav")